In [1]:
# %% [code]
import os, glob, shutil
import pandas as pd
import ee
import time
import requests
import huggingface_hub
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
huggingface_key = user_secrets.get_secret("huggingface_token")

project = "landslide-identification-nepal" #The google earth engine project name
input_csv = "/kaggle/input/datasets/sanjayashrestha123/landslide-reproted/landslides_from_2018_to_2026.csv" #location of .csv containing landslide incidents
upload_batch = 100
repo_id = "sasudo2/landslides"
DATASET_REVISION = "main"  # set to e.g. "test-v1" to push a separate dataset revision/branch (no effect on main)

DOWNLOAD_DIR = '/kaggle/working/downloads'
os.makedirs(DOWNLOAD_DIR, exist_ok=True)   # folder name in your Google Drive
api = huggingface_hub.HfApi(token = huggingface_key)
api.create_repo(repo_id="sasudo2/landslides", repo_type="dataset", exist_ok=True)


gee_key = "/kaggle/input/datasets/sanjayashrestha123/gee-key/landslide-identification-nepal-cccd90850069.json"
service_account = 'kaggle-import@landslide-identification-nepal.iam.gserviceaccount.com'

credentials = ee.ServiceAccountCredentials(service_account, gee_key)

try:
    ee.Initialize(credentials, project=project)
except Exception as e:
    print("EE initialization failed.")
    raise e

csv_filename = input_csv
df = pd.read_csv(csv_filename)
df = df.iloc[721:]
df['incident_on'] = pd.to_datetime(df['incident_on'])

MAX_AOI_DEG = 0.1

/tmp/ipykernel_16/3782228842.py:39: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['incident_on'] = pd.to_datetime(df['incident_on'])


In [ ]:
# %% [markdown]
#### Why this rewrite (fixes the "cartoonish" median composite)
The previous version called `collection.median()` over an **18-month** window on both
sides of the incident date. A median over ~hundreds of scenes blends many different
atmospheric/seasonal/illumination states into one flat, cartoon-like image and smears
away the actual post-event scene.

Per `landslide_workflow.md` (Stage 0) we now instead:
1. Pick the **best single acquisition date** (or a single same-day pass) closest to the
   incident on the pre- and post-event side.
2. For each selected date, pull the **exact MGRS tiles** that the AOI overlaps and
   **mosaic them spatially** (neighbouring tiles from the same overpass day) — this is a
   spatial stitch, NOT a temporal composite, so the scene looks continuous and real.
3. Download all 12 S2 reflectance bands (+ SCL) so the indices from the workflow
   (NDVI, NDWI, BSI, NBR) can be computed later, plus DEM slope & aspect.
4. Optionally pull paired pre/post Sentinel-1 GRD (VV/VH, same orbit direction)
   for the SAR amplitude-ratio change cue.

File naming and HF structure are **unchanged**:
`incident_<ID>/incident_<ID>_{before,after,slope,aspect}.tif` (+ `_sar_pre` / `_sar_post` when Sentinel-1 available),
uploaded with `allow_patterns="*.tif"` exactly as before.

In [ ]:
# %% [code]
def mask_s2_clouds(image):
    """Keep only clear/valid SCL classes (same semantics as before).
    Classes: 2=dark, 4=veg,5=not-veg,6=water,7=uncl,11=snow. We drop cloud/shadow.
    """
    scl = image.select('SCL')
    clean_mask = (scl.eq(2).bitwiseOr(scl.eq(4))
                           .bitwiseOr(scl.eq(5))
                           .bitwiseOr(scl.eq(6))
                           .bitwiseOr(scl.eq(7))
                           .bitwiseOr(scl.eq(11)))
    return image.updateMask(clean_mask)

def clamp_aoi(min_lon, min_lat, max_lon, max_lat):
    lon_span = max_lon - min_lon
    lat_span = max_lat - min_lat
    if lon_span <= MAX_AOI_DEG and lat_span <= MAX_AOI_DEG:
        return min_lon, min_lat, max_lon, max_lat
    cx = (min_lon + max_lon) / 2
    cy = (min_lat + max_lat) / 2
    half = MAX_AOI_DEG / 2
    return cx - half, cy - half, cx + half, cy + half

def download_image(image, aoi, incident_id, filename, scale=10, max_retries=5):
    os.makedirs(f"{DOWNLOAD_DIR}/incident_{incident_id}", exist_ok=True)
    filepath = f'{DOWNLOAD_DIR}/incident_{incident_id}/{filename}.tif'
    for attempt in range(1, max_retries + 1):
        try:
            url = image.getDownloadURL({
                'scale': scale,
                'region': aoi,
                'format': 'GeoTIFF',
                'crs': 'EPSG:4326',
            })
            response = requests.get(url, stream=True, timeout=300)
            if response.status_code == 429:
                wait = 15 * attempt
                print(f"  429 on {filename}, retry {attempt}/{max_retries} after {wait}s")
                time.sleep(wait)
                continue
            response.raise_for_status()
            with open(filepath, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"Downloaded: {filepath}")
            return
        except Exception as e:
            if attempt == max_retries:
                print(f"Failed to download {filename}: {e}")
                return
            wait = 15 * attempt
            print(f"  error on {filename}, retry {attempt}/{max_retries} after {wait}s: {e}")
            time.sleep(wait)
    print(f"Gave up on {filename} after {max_retries} retries")



In [ ]:
# %% [code]
def best_scene(collection, label):
    """Pick the least-cloudy single image from a filtered collection.

    For S2: returns the single best acquisition date (one overpass) so the
    result is a same-day spatial mosaic of the AOI's MGRS tiles — no temporal
    blending, no cartoon artifact.
    """
    count = collection.size().getInfo()
    if count == 0:
        print(f"  {label}: no scenes found.")
        return None
    best = collection.sort('CLOUDY_PIXEL_PERCENTAGE').first()
    return best

def get_single_sar_image(s1_collection, label):
    count = s1_collection.size().getInfo()
    if count == 0:
        print(f"  SAR {label}: no scenes found.")
        return None
    best = s1_collection.sort('system:time_start').first()
    return best

def submit_landslide_export(incident_id,
                            pre_days=180, post_days=45,
                            include_sar=True):
    # Cast to int so the hub folder is always integer-named (e.g. incident_74277),
    # consistent with candidate_detection.ipynb which also casts to int.
    incident_id = int(incident_id)
    """Download single-date pre/post S2 + DEM slope/aspect + paired S1 SAR.
    See landslide_workflow.md Stage 0.2-0.4.
    """
    row = df[df['id'] == incident_id]
    if row.empty:
        print(f"ID {incident_id} not found.")
        return
    row = row.iloc[0]
    incident_date = row['incident_on']
    c_min_lon, c_min_lat, c_max_lon, c_max_lat = clamp_aoi(
        row['min_lon'], row['min_lat'], row['max_lon'], row['max_lat'])
    aoi = ee.Geometry.Rectangle([c_min_lon, c_min_lat, c_max_lon, c_max_lat])

    before_start = (incident_date - pd.DateOffset(days=pre_days)).strftime('%Y-%m-%d')
    before_end   = (incident_date - pd.DateOffset(days=5)).strftime('%Y-%m-%d')
    after_start  = (incident_date + pd.DateOffset(days=5)).strftime('%Y-%m-%d')
    after_end    = (incident_date + pd.DateOffset(days=post_days)).strftime('%Y-%m-%d')

    print(f"\nChecking ID {incident_id}: {row['title']}")

    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(aoi)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 70))
            .map(mask_s2_clouds))

    bands = ['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B9','B11','B12','SCL']

    before_col = s2.filterDate(before_start, before_end)
    after_col  = s2.filterDate(after_start, after_end)

    before_img = best_scene(before_col, 'before')
    after_img  = best_scene(after_col, 'after')

    if before_img is None or after_img is None:
        print(f"Skipping ID {incident_id} — missing pre or post scene.")
        return

    print(f"  before date: {before_img.date().format().getInfo()}")
    print(f"  after  date: {after_img.date().format().getInfo()}")

    download_image(before_img.select(bands).clip(aoi), aoi, incident_id, f'incident_{incident_id}_before', scale=10)
    download_image(after_img.select(bands).clip(aoi),  aoi, incident_id, f'incident_{incident_id}_after',  scale=10)

    # DEM derivatives (Stage 0.4 of landslide_workflow.md)
    dem = ee.Image('USGS/SRTMGL1_003')
    slope = ee.Terrain.slope(dem).clip(aoi)
    aspect = ee.Terrain.aspect(dem).clip(aoi)
    download_image(slope,  aoi, incident_id, f'incident_{incident_id}_slope', scale=30)
    download_image(aspect, aoi, incident_id, f'incident_{incident_id}_aspect', scale=30)

    # Optional Sentinel-1 GRD (Stage 0.3): SAME orbit direction for pre & post,
    # otherwise the backscatter difference is dominated by geometry, not real change.
    if include_sar:
        try:
            s1_base = (ee.ImageCollection('COPERNICUS/S1_GRD')
                        .filterBounds(aoi)
                        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                        .filter(ee.Filter.eq('instrumentMode', 'IW')))
            # Narrow windows aligned to the optical pre/post span (NOT the 180d pre window)
            sar_pre_start  = (incident_date - pd.DateOffset(days=45)).strftime('%Y-%m-%d')
            sar_post_end   = (incident_date + pd.DateOffset(days=post_days)).strftime('%Y-%m-%d')
            s1_pre  = s1_base.filterDate(sar_pre_start, before_end)
            s1_post = s1_base.filterDate(after_start, sar_post_end)
            # pick the orbit pass (ASCENDING/DESCENDING) present in BOTH windows
            pre_pass  = s1_pre.aggregate_array('orbitProperties_pass').getInfo()
            post_pass = s1_post.aggregate_array('orbitProperties_pass').getInfo()
            common_passes = set(pre_pass) & set(post_pass)
            if not common_passes:
                raise ValueError('no common orbit pass between pre and post S1')
            orbit = sorted(common_passes)[0]
            s1_pre  = s1_pre.filter(ee.Filter.eq('orbitProperties_pass', orbit))
            s1_post = s1_post.filter(ee.Filter.eq('orbitProperties_pass', orbit))
            s1_img_pre  = get_single_sar_image(s1_pre, 'pre')
            s1_img_post = get_single_sar_image(s1_post, 'post')
            if s1_img_pre is not None and s1_img_post is not None:
                sar_bands = ['VV','VH']
                download_image(s1_img_pre.select(sar_bands).clip(aoi), aoi, incident_id,
                               f'incident_{incident_id}_sar_pre', scale=10)
                download_image(s1_img_post.select(sar_bands).clip(aoi), aoi, incident_id,
                               f'incident_{incident_id}_sar_post', scale=10)
                print(f"  SAR pre date: {s1_img_pre.date().format().getInfo()}")
                print(f"  SAR post date: {s1_img_post.date().format().getInfo()}")
        except Exception as e:
            print(f"  SAR fetch skipped: {e}")


In [ ]:
# %% [code]
def flush_uploads():
    try:
        api.upload_folder(
            folder_path=DOWNLOAD_DIR,
            repo_id="sasudo2/landslides",
            repo_type="dataset",
            revision=DATASET_REVISION,
            allow_patterns="*.tif",
        )
    except Exception as e:
        print(f"!!!Upload failed, keeping local files: {e}!!!")
        return

    for subdir in glob.glob(f"{DOWNLOAD_DIR}/*/"):
        shutil.rmtree(subdir)

    print(f"Uploaded and cleared {DOWNLOAD_DIR}")

from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_WORKERS = 2  # keep low to avoid GEE 429 rate limits

def process_incident(inc_id):
    submit_landslide_export(inc_id)
    return inc_id

upload_count = 0
incident_ids = df['id'].iloc[:].tolist()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_incident, inc_id): inc_id for inc_id in incident_ids}

    for future in as_completed(futures):
        inc_id = futures[future]
        try:
            future.result()
        except Exception as e:
            print(f"Incident {inc_id} failed: {e}")

        upload_count += 1
        if upload_count % upload_batch == 0:
            flush_uploads()

if upload_count % upload_batch != 0:
    flush_uploads()